In [1]:
import os
import re
import pandas as pd
import numpy as np
import spacy
import joblib
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, util
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [2]:
print("Tizim yuklanmoqda...")
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

print("DL Transformer modeli yuklanmoqda (Hugging Face)...")
dl_model = SentenceTransformer('all-MiniLM-L6-v2')

Tizim yuklanmoqda...
DL Transformer modeli yuklanmoqda (Hugging Face)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:
def clean_texts(text_series):
    raw = (
        text_series.fillna("")
        .astype(str)
        .apply(lambda t: re.sub(r"[^a-zA-Z\s]", "", t).lower().strip())
    )
    cleaned = []
    for doc in nlp.pipe(raw.tolist(), batch_size=256):
        tokens = [tok.lemma_ for tok in doc if not tok.is_stop and tok.lemma_.strip()]
        cleaned.append(" ".join(tokens))
    return pd.Series(cleaned, index=text_series.index)


def word_overlap_ratio(a, b):
    set_a, set_b = set(a.split()), set(b.split())
    if not set_a or not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)


In [5]:
def main():
    print("\n=== 1-TUGMA: SMART HR RECRUITER PIPELINE ===")

    dataset_path = "resume_data.csv"

    if not os.path.exists(dataset_path):
        print(f"Xato: {dataset_path} fayli topilmadi!")
        return

    df = pd.read_csv(dataset_path)
    print(f"Dataset yuklandi. Jami qatorlar: {len(df)}")


    print("\n[1/3] NLP: Matnli ustunlar tozalanmoqda...")
    df["cleaned_skills"] = clean_texts(df["skills"])
    df["cleaned_skills_required"] = clean_texts(df["skills_required"])

    both_empty = (df["cleaned_skills"] == "") & (df["cleaned_skills_required"] == "")
    print(f"Diqqat: {both_empty.sum()} qatorda ikkala ustun ham bo'sh (natija ishonchsiz bo'lishi mumkin).")


    print("\n[2/3] DL: Sentence Embeddings orqali moslik foizi hisoblanmoqda...")

    emb_skills = dl_model.encode(
        df["cleaned_skills"].tolist(), batch_size=64, convert_to_tensor=True, show_progress_bar=True
    )
    emb_required = dl_model.encode(
        df["cleaned_skills_required"].tolist(), batch_size=64, convert_to_tensor=True, show_progress_bar=True
    )

    cos_scores = util.cos_sim(emb_skills, emb_required).diagonal().cpu().numpy()
    df["dl_semantic_score"] = cos_scores
    df.loc[both_empty, "dl_semantic_score"] = 0.0

    print("DL bosqichi yakunlandi. Semantik ballar hisoblab chiqildi.")


    print("\nQo'shimcha xususiyatlar (feature) hisoblanmoqda...")
    tqdm.pandas(desc="So'z darajasidagi moslik")
    df["word_overlap_score"] = df.progress_apply(
        lambda r: word_overlap_ratio(r["cleaned_skills"], r["cleaned_skills_required"]), axis=1
    )
    df["skills_len"] = df["cleaned_skills"].str.split().apply(len)
    df["skills_required_len"] = df["cleaned_skills_required"].str.split().apply(len)

    print("\n[3/3] ML: Random Forest Regressor modeli o'qitilmoqda...")

    feature_cols = [
        "dl_semantic_score",
        "word_overlap_score",
        "skills_len",
        "skills_required_len",
    ]
    X = df[feature_cols]
    y = df["matched_score"]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    ml_model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    ml_model.fit(X_train, y_train)

    train_r2 = ml_model.score(X_train, y_train)
    test_r2 = ml_model.score(X_test, y_test)
    test_pred = ml_model.predict(X_test)
    mae = mean_absolute_error(y_test, test_pred)
    rmse = np.sqrt(mean_squared_error(y_test, test_pred))

    print(f"Model muvaffaqiyatli o'qitildi!")
    print(f"Train R2 Score: {train_r2:.4f}")
    print(f"Test R2 Score:  {test_r2:.4f}")
    print(f"Test MAE:       {mae:.4f}")
    print(f"Test RMSE:      {rmse:.4f}")

    print("\nFeature ahamiyati (importance):")
    for name, imp in sorted(zip(feature_cols, ml_model.feature_importances_), key=lambda x: -x[1]):

        print(f"  {name}: {imp:.4f}")
    print("\nModellarni faylga saqlash...")
    os.makedirs("models", exist_ok=True)

    joblib.dump(ml_model, "models/hr_train.joblib")
    joblib.dump(feature_cols, "models/feature_cols.joblib")
    print("Tayyor ML model 'models/hr_train.joblib' manziliga saqlandi.")
    print("Feature ro'yxati 'models/feature_cols.joblib' manziliga saqlandi.")
    print("(DL model FastAPI backendda nomi bo'yicha qayta yuklanadi: 'all-MiniLM-L6-v2')")
    print("1-Tugma backend tayyorlash uchun 100% tayyor!")


if __name__ == "__main__":
    main()




=== 1-TUGMA: SMART HR RECRUITER PIPELINE ===
Dataset yuklandi. Jami qatorlar: 9544

[1/3] NLP: Matnli ustunlar tozalanmoqda...
Diqqat: 20 qatorda ikkala ustun ham bo'sh (natija ishonchsiz bo'lishi mumkin).

[2/3] DL: Sentence Embeddings orqali moslik foizi hisoblanmoqda...


Batches:   0%|          | 0/150 [00:00<?, ?it/s]

Batches:   0%|          | 0/150 [00:00<?, ?it/s]

DL bosqichi yakunlandi. Semantik ballar hisoblab chiqildi.

Qo'shimcha xususiyatlar (feature) hisoblanmoqda...


So'z darajasidagi moslik: 100%|██████████| 9544/9544 [00:00<00:00, 48900.80it/s]



[3/3] ML: Random Forest Regressor modeli o'qitilmoqda...
Model muvaffaqiyatli o'qitildi!
Train R2 Score: 0.8551
Test R2 Score:  0.2740
Test MAE:       0.1072
Test RMSE:      0.1417

Feature ahamiyati (importance):
  dl_semantic_score: 0.4147
  skills_len: 0.2894
  skills_required_len: 0.2094
  word_overlap_score: 0.0864

Modellarni faylga saqlash...
Tayyor ML model 'models/hr_train.joblib' manziliga saqlandi.
Feature ro'yxati 'models/feature_cols.joblib' manziliga saqlandi.
(DL model FastAPI backendda nomi bo'yicha qayta yuklanadi: 'all-MiniLM-L6-v2')
1-Tugma backend tayyorlash uchun 100% tayyor!
